# FarmWise AI: Real Crop Recommendation Model Training
---
This notebook trains the core **Crop Advisor ML Model** using **REAL agricultural data** from the Kaggle Crop Recommendation Dataset. 

**Dataset Features:**
- `N`: Nitrogen content ratio in soil
- `P`: Phosphorous content ratio in soil
- `K`: Potassium content ratio in soil
- `temperature`: temperature in degree Celsius
- `humidity`: relative humidity in %
- `ph`: ph value of the soil
- `rainfall`: rainfall in mm
- **Target Label**: Exact crop recommended based on best conditions

**Output:** A trained `crop_model.pkl` pipeline containing both the model and the label encoders.

In [ ]:
!pip install pandas numpy scikit-learn seaborn matplotlib requests

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

### 1. Download and Load Real Data

In [ ]:
# Downloading directly from public repository (Authored by Kaggle Dataset Creator)
url = "https://raw.githubusercontent.com/Atharva-Ingle/Crop-Recommendation-Dataset/master/Crop_recommendation.csv"
df = pd.read_csv(url)
print(f"Loaded dataset shape: {df.shape}")
df.head()

In [ ]:
# Check class distribution (Real dataset has 22 crops, 100 samples each)
plt.figure(figsize=(12, 6))
sns.countplot(y=df['label'])
plt.title('Distribution of Crops in Dataset')
plt.show()

### 2. Prepare Features and Encodings

In [ ]:
# The target label is categorical, we encode it to integers
le_crop = LabelEncoder()
df['label_encoded'] = le_crop.fit_transform(df['label'])

FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[FEATURES]
y = df['label_encoded']

# 80/20 Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

### 3. Train the Model
Using RandomForest for robust ensemble learning. Decision Trees overfit too easily on this NPK matrix.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

### 4. Evaluate Real World Accuracy

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc:.2%}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le_crop.classes_))

### 5. Export for FastAPI Backend

In [ ]:
os.makedirs('../backend/app/models', exist_ok=True)
export_path = '../backend/app/models/crop_model.pkl'

bundle = {
    'model': model,
    'le_crop': le_crop,
    'features': FEATURES,
    'accuracy': acc
}

with open(export_path, 'wb') as f:
    pickle.dump(bundle, f)

print(f"\u2705 Model successfully saved to {export_path}")